# 01-Kim: 数据读入 + 质量控制

## 唯一参数入口（四组）

本 cell 是本阶段唯一改参数的地方，分四组：

- **数据源与版本**：改哪批数据、输出第几版
- **QC 与科学阈值**：会改变纳入哪些细胞/基因的科研判断
- **方法开关**：开/关某步分析
- **输出与运行标识**：RUN_ID 与复现信息

提示：改参数请新开 RUN_ID，不覆盖旧 run；每个科学参数下方注释给出「是什么/默认依据/调大调小影响/何时该改」。



In [ ]:
# === PARAMS ===

# ---------- 组一 · 数据源与版本 ----------
MANIFEST_PATH = "data/kim/manifest.yaml"

# ---------- 组二 · QC 与科学阈值 ----------
#（QC 策略 / MAD / 基因过滤 / 固定阈值 / doublet 三态阈值——都会改变纳入哪些细胞或基因）
# 是什么：自适应（按数据分布定阈，跨样本稳健）vs 固定阈值（人工写死上下限，数据异质时易误删）
# 默认依据：adaptive 是推荐默认，无需预知数据基线
# 调大调小影响：调为 fixed = 需手动指定 MIN_GENES/MAX_GENES/MIN_COUNTS/MAX_PCT_MT
# 何时该改：已知明确生物学阈值或需与旧流程对齐时改为 fixed
QC_STRATEGY = "adaptive"
# 是什么：自适应阈值的中位绝对偏差倍数
# 默认依据：3（约覆盖正态分布 99% 的细胞）
# 调大调小影响：调大=更宽松、留更多细胞（含低质量）；调小=更严、多删细胞（含真细胞）
# 何时该改：污染重时调小（如 2~2.5），稀有细胞多时调大（如 4~5）
N_MAD = 3
# 是什么：按样本各自算 MAD vs 全局统一
# 默认依据：True（样本间深度差异大时更公平，每个样本用自己的基线）
# 调大调小影响：改 False=全局单一阈值（可能系统性偏向高深度样本，对浅样本过严或对深样本过松）
# 何时该改：仅需与旧全局行为对齐时改为 False
PER_SAMPLE_MAD = True
# 是什么：基因保留最低表达细胞数
# 默认依据：3（过滤背景噪声基因，跨数据集通用）
# 调大调小影响：调大=删更多稀疏基因（可能丢稀有细胞类型 marker）；调小=留更多噪声基因
# 何时该改：研究稀有细胞类型时可调小（如 1~2），极大数据集可调大
MIN_CELLS_PER_GENE = 3
DOUBLET_SCORE_THRESHOLD = None  # None=使用 scrublet 自动阈值 | float（如 0.25）=手动覆盖

# doublet 三态定级参数（决策8：singlet/uncertain/doublet）
# 含义：高于 HIGH→高置信 doublet（默认排除）；低于 LOW→高置信 singlet；介于 [LOW,HIGH]→uncertain（默认保留）
# 默认值：HIGH=None 用 scrublet 自动阈值；LOW=None 自动派生为 HIGH*0.5
# 调整：数据质量差/双峰不明显时可手动指定 HIGH；调整风险：过高的 HIGH 漏检 doublet，过低的 LOW 误删 singlet
DOUBLET_SCORE_HIGH = None   # 高置信 doublet 阈值；None=用 scrublet 自动阈值 threshold_；> HIGH → doublet
DOUBLET_SCORE_LOW = None    # 低置信 singlet 上界；None=按 HIGH 派生（=HIGH*0.5）；< LOW → singlet；介于 [LOW,HIGH] → uncertain
DOUBLET_RATE_ALERT_HIGH = 0.40     # 单样本 doublet 预测比例上限；> 此值触发 needs_review（异常高比例警报）；默认 0.40 适合 10x 3' 数据；Smart-seq2 可收紧至 0.15
DOUBLET_RATE_ALERT_LOW = None      # doublet 预测比例下限（可选）；None=不设下限告警；调整：若预期数据质量极高可设 0.01 以捕获 scrublet 异常
DOUBLET_MIN_CELLS = 50             # 样本最低细胞数；< 此值无法稳定定阈值，记 needs_review 而非静默跳过

# --- 固定阈值（仅 QC_STRATEGY="fixed" 时生效）---
# 以下四项仅在 QC_STRATEGY="fixed" 时生效，adaptive 模式下为占位
# 是什么：人工写死的 QC 上下限，不依赖数据自身分布
# 默认依据：通用类器官/10x 数据经验值，待根据数据集实测分布校准
# 调大调小影响：调宽=留更多边缘细胞；调窄=更严格过滤
# 何时该改：QC_STRATEGY="fixed" 时根据数据集 QC 分布图设定；当前 adaptive 无需修改
MIN_GENES  = 200  # TODO: 仅 QC_STRATEGY="fixed" 时生效，当前 adaptive 模式下仅占位
MAX_GENES  = 6000  # TODO: 仅 QC_STRATEGY="fixed" 时生效，当前 adaptive 模式下仅占位
MIN_COUNTS = 500  # TODO: 仅 QC_STRATEGY="fixed" 时生效，当前 adaptive 模式下仅占位
MAX_PCT_MT = 20  # TODO: 仅 QC_STRATEGY="fixed" 时生效，当前 adaptive 模式下仅占位

# ---------- 组三 · 方法开关 ----------
#（开/关某步分析或改变纳入策略；不是连续阈值）
# 是什么：是否做环境 RNA 校正（SoupX）
# 默认依据：False（Kim 为类器官 h5ad，无 raw droplet matrix，无法估计环境 RNA）
# 调大调小影响：开 True 需要 manifest 提供 raw_matrix_path，对 X 做 counts 减法
# 何时该改：仅当拿到含空液滴的 raw 10x 矩阵时才改为 True
SOUPX_ENABLED = False
# 是什么：scrublet 预期双细胞率，触发 per-sample 双细胞检测
# 默认依据：None（不跑 scrublet，全标 singlet）
# 调大调小影响：设为 (0,1) 才触发检测；调高/低=改变 scrublet 内部模拟双细胞比例与阈值
# 何时该改：按上样细胞数查 10x 表（约 0.008/千细胞）；类器官无 10x 上样背景通常不设
EXPECTED_DOUBLET_RATE = None
DOUBLET_UNCERTAIN_INCLUDE = True   # uncertain 细胞是否纳入下游（True=标记但纳入，False=排除）；调整风险：True 可能漏排潜在问题细胞，False 损失数据量
# 是什么：是否计算细胞周期 S/G2M 评分
# 默认依据：True（周期效应是重要技术协变量，下游可用评分回归校正）
# 调大调小影响：关闭=跳过 S/G2M 打分，下游无法回归周期效应
# 何时该改：确认周期非关注变量且想省算力时改为 False
SCORE_CELL_CYCLE = True
# 是什么：是否标记血红蛋白基因（红细胞污染指标）
# 默认依据：False（类器官无 RBC 污染，标记无意义）
# 调大调小影响：开 True=增加 flag_hb 列，可能在使用人血红蛋白基因列表的非人数据上误标
# 何时该改：组织活检来源数据可能含血污染时改为 True
FLAG_HEMOGLOBIN = False
# 是什么：是否标记应激基因（解离诱导的即时早期基因与热休克蛋白）
# 默认依据：True（任何组织解离都会诱导应激，标记供下游评估解离质量）
# 调大调小影响：关闭=跳过应激标记，下游无法评估解离批次效应
# 何时该改：确认应激非关注变量时改为 False
FLAG_STRESS_GENES = True

# ---------- 组四 · 输出与运行标识 ----------
RUN_ID = "01-kim-v1-run001"  # 每次调参改用新 ID，禁止覆盖旧 run
RUN_ROOT = "results/runs"
OUTPUT_FILENAME = "01_kim_v1.h5ad"
OUTPUT_VERSION = 1
RANDOM_SEED = 42



In [ ]:
# === 启动脚手架：一行替代原来 20 行样板代码 ===
# init() 完成：找项目根 + BLAS 线程设置 + sys.path 注入 + print 诊断结果
# 遵循 "platform 决策可见" 原则——决策结果直接在 cell 输出区显示

from scrna_integration.bootstrap import init
_root = init()

import os
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)

import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import shutil
import warnings
import yaml
import gc
from pathlib import Path
from scipy.stats import median_abs_deviation
from scrna_integration.run_contract import (
    atomic_write_json, collect_runtime_provenance, determine_stage_status, prepare_run,
    promote_run, sha256_file, snapshot_effective_parameters, validate_expression_contract,
)

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

import importlib.metadata
print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

# platform 决策可见：打印 R 环境状态
from scrna_integration.platform import check_r_available
RSCRIPT_BIN, R_AVAILABLE = check_r_available()
print(f"Rscript    : {'可用' if R_AVAILABLE else '不可用'} ({RSCRIPT_BIN})")

## Preflight 执行前校验

在读入数据前集中校验输入与参数，任一不合法立即 raise 明确错误，避免跑到中途才崩或静默产出错误结果。



In [ ]:
# === Preflight：执行前校验 ===
# 在读入数据前集中校验输入与参数，任一不合法立即 raise 明确错误，避免跑到中途才崩

# yaml 与 Path 已由上方「启动脚手架」cell 导入，此处直接使用

# --- 1. manifest 路径存在 ---
if not Path(MANIFEST_PATH).is_file():
    raise FileNotFoundError(f"manifest 文件不存在: {MANIFEST_PATH}")

# --- 2. manifest 可解析且含必需字段 ---
with open(MANIFEST_PATH) as f:
    _pf_manifest = yaml.safe_load(f)
if "input" not in _pf_manifest:
    raise ValueError(f"manifest 缺少必需字段 'input': {MANIFEST_PATH}")
if "source_dataset" not in _pf_manifest:
    raise ValueError(f"manifest 缺少必需字段 'source_dataset': {MANIFEST_PATH}")

# --- 3. 输入格式与文件 ---
_input_block = _pf_manifest["input"]
if _input_block.get("format") != "h5ad":
    raise ValueError(
        f"Kim 数据集预期 h5ad 格式，manifest 声明为 {_input_block.get('format')}: {MANIFEST_PATH}"
    )
_input_path = _input_block.get("path")
if not _input_path:
    raise ValueError(f"manifest input 缺少 'path': {MANIFEST_PATH}")
if not Path(_input_path).is_file():
    raise FileNotFoundError(f"输入数据文件不存在: {_input_path}")

# --- 4. 关键 PARAMS 类型/取值合法 ---
if QC_STRATEGY not in {"adaptive", "fixed"}:
    raise ValueError(f"QC_STRATEGY 必须为 'adaptive' 或 'fixed'，当前: {QC_STRATEGY}")
if not isinstance(N_MAD, (int, float)) or N_MAD <= 0:
    raise ValueError(f"N_MAD 必须为正数，当前: {N_MAD}")
if not isinstance(RANDOM_SEED, int):
    raise ValueError(f"RANDOM_SEED 必须为整数，当前: {type(RANDOM_SEED).__name__}")
if not isinstance(OUTPUT_VERSION, int) or OUTPUT_VERSION < 1:
    raise ValueError(f"OUTPUT_VERSION 必须为 >=1 的整数，当前: {OUTPUT_VERSION}")
if not isinstance(MIN_CELLS_PER_GENE, int) or MIN_CELLS_PER_GENE < 0:
    raise ValueError(f"MIN_CELLS_PER_GENE 必须为非负整数，当前: {MIN_CELLS_PER_GENE}")

# bool 参数类型检查：这些参数控制分析分支走向，非 bool 会导致 if 判断行为异常
_bool_params = {
    "PER_SAMPLE_MAD": PER_SAMPLE_MAD,
    "SOUPX_ENABLED": SOUPX_ENABLED,
    "SCORE_CELL_CYCLE": SCORE_CELL_CYCLE,
    "FLAG_HEMOGLOBIN": FLAG_HEMOGLOBIN,
    "FLAG_STRESS_GENES": FLAG_STRESS_GENES,
    "DOUBLET_UNCERTAIN_INCLUDE": DOUBLET_UNCERTAIN_INCLUDE,
}
for _name, _val in _bool_params.items():
    if not isinstance(_val, bool):
        raise ValueError(f"{_name} 必须为 bool，当前: {type(_val).__name__}")

# doublet 参数范围检查：阈值边界非法会导致三态定级逻辑异常
if not isinstance(DOUBLET_MIN_CELLS, int) or DOUBLET_MIN_CELLS <= 0:
    raise ValueError(f"DOUBLET_MIN_CELLS 必须为正整数，当前: {DOUBLET_MIN_CELLS}")
if not (0 < DOUBLET_RATE_ALERT_HIGH <= 1):
    raise ValueError(
        f"DOUBLET_RATE_ALERT_HIGH 必须在 (0, 1]，当前: {DOUBLET_RATE_ALERT_HIGH}"
    )
if DOUBLET_RATE_ALERT_LOW is not None:
    if not (0 <= DOUBLET_RATE_ALERT_LOW <= DOUBLET_RATE_ALERT_HIGH):
        raise ValueError(
            f"DOUBLET_RATE_ALERT_LOW ({DOUBLET_RATE_ALERT_LOW}) 必须在 "
            f"[0, DOUBLET_RATE_ALERT_HIGH ({DOUBLET_RATE_ALERT_HIGH})] 区间内"
        )
if EXPECTED_DOUBLET_RATE is not None:
    if not (0 < EXPECTED_DOUBLET_RATE < 1):
        raise ValueError(
            f"EXPECTED_DOUBLET_RATE 必须在 (0, 1)，当前: {EXPECTED_DOUBLET_RATE}"
        )

# doublet 分值范围检查（概率值应在 [0,1]）
for _dname, _dval in [
    ("DOUBLET_SCORE_THRESHOLD", DOUBLET_SCORE_THRESHOLD),
    ("DOUBLET_SCORE_HIGH", DOUBLET_SCORE_HIGH),
    ("DOUBLET_SCORE_LOW", DOUBLET_SCORE_LOW),
]:
    if _dval is not None and not (0 <= _dval <= 1):
        raise ValueError(f"{_dname} 必须在 [0, 1]，当前: {_dval}")

# doublet 阈值序：LOW <= HIGH，否则三态区间倒置导致定级语义混乱
if DOUBLET_SCORE_HIGH is not None and DOUBLET_SCORE_LOW is not None:
    if DOUBLET_SCORE_LOW > DOUBLET_SCORE_HIGH:
        raise ValueError(
            f"DOUBLET_SCORE_LOW ({DOUBLET_SCORE_LOW}) > DOUBLET_SCORE_HIGH "
            f"({DOUBLET_SCORE_HIGH})，三态阈值区间倒置"
        )

# --- 5. SOUPX 开关与数据集匹配 ---
if SOUPX_ENABLED:
    _raw_path = _pf_manifest.get("raw_matrix_path") or _input_block.get("raw_matrix_path")
    if not _raw_path:
        raise ValueError(
            "SOUPX_ENABLED=True 但 manifest 无 raw_matrix_path；"
            "Kim 类器官默认 SOUPX_ENABLED=False，如需开启请先在 manifest 中配置 raw_matrix_path"
        )
else:
    print("SoupX 已按 Kim 数据集特点关闭（类器官无 raw droplet matrix）")

# --- 6. 生效参数速览 ---
print("===== 生效参数速览 =====")
print(f"  数据源     : {MANIFEST_PATH}")
print(f"  格式       : {_input_block.get('format')}")
print(f"  source_dataset: {_pf_manifest.get('source_dataset')}")
print(f"  QC 策略    : {QC_STRATEGY}（N_MAD={N_MAD}, per_sample={PER_SAMPLE_MAD}）")
print(f"  SoupX      : {'开启' if SOUPX_ENABLED else '关闭'}")
_dt_enabled = EXPECTED_DOUBLET_RATE is not None
print(f"  doublet检测: {'开启（rate=' + str(EXPECTED_DOUBLET_RATE) + '）' if _dt_enabled else '关闭'}")
print(f"  OUTPUT_VERSION: {OUTPUT_VERSION}")
print(f"  RUN_ID     : {RUN_ID}")
print("Preflight 校验通过。")



In [ ]:
# === 数据读入：替代原来的 read_with_manifest ===
# 按透明性铁律，io.py 的通用调度逻辑拆回 cell：
# 不再靠泛化字典做 obs_mapping / value_mapping / 格式分发，
# 而是针对 Kim 数据集的实际字段直接写处理逻辑。
# 眼前可见、可改——每个字段映射的来源和去向一目了然。


# ---- 1. 加载并校验 manifest ----
print(f"正在加载 manifest: {MANIFEST_PATH}")
with open(MANIFEST_PATH) as f:
    manifest = yaml.safe_load(f)

# 断言必要字段存在
assert "input" in manifest, "manifest 缺少 'input' 字段"
assert "source_dataset" in manifest, "manifest 缺少 'source_dataset' 字段"

input_block = manifest["input"]
fmt = input_block["format"]
data_path = input_block["path"]
source_dataset = manifest["source_dataset"]

print(f"来源数据集 : {source_dataset}")
print(f"数据格式   : {fmt}")
print(f"数据路径   : {data_path}")

# ---- 2. 读入数据（Kim 数据集是 h5ad 格式，一行 sc.read_h5ad）----
assert fmt == "h5ad", f"Kim 数据集预期 h5ad 格式，实际 {fmt}"
adata = sc.read_h5ad(data_path)
adata.obs["source_dataset"] = source_dataset
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")

# ---- 3. 应用 obs_mapping（列重命名：外部名 → 框架标准名）----
# Kim 数据集的映射操作直接写在 cell 里，不靠泛化字典隐藏
obs_mapping = manifest.get("obs_mapping", {})
if obs_mapping:
    # manifest：{标准名: 外部名} → 反转：{外部名: 标准名}
    rename_dict = {v: k for k, v in obs_mapping.items()}
    print(f"列重命名: {rename_dict}")
    # 只重命名存在的列
    existing = {k: v for k, v in rename_dict.items() if k in adata.obs.columns}
    adata.obs = adata.obs.rename(columns=existing)

# ---- 4. 应用 value_mapping（值统一）----
value_mapping = manifest.get("value_mapping", {})
for col, mapping in value_mapping.items():
    if col in adata.obs.columns:
        adata.obs[col] = adata.obs[col].astype(str).map(
            lambda x, m=mapping: m.get(x, x)
        )
        print(f"  {col} 值映射: {mapping}")

# ---- 5. 基因 ID 格式检查（Kim 数据集是 symbol，无需同步）----
gene_id_format = input_block.get("gene_id_format", "symbol")
print(f"基因 ID 格式: {gene_id_format}")

# ---- 6. 打印最终状态 ----
print(f"最终 obs 列: {list(adata.obs.columns)}")
print(f"var 列: {list(adata.var.columns)}")
if "disease" in adata.obs.columns:
    print(f"disease 取值: {dict(adata.obs['disease'].value_counts())}")
# ---- 7. 建立 counts 契约（决策 1/2）----
# layers["counts"] 是进入框架后唯一 counts 权威位置，一经建立不得被覆盖
adata.layers["counts"] = adata.X.copy()
# 确保 CSR float32（内存纪律）
if not sp.issparse(adata.layers["counts"]) or adata.layers["counts"].dtype != np.float32:
    adata.layers["counts"] = sp.csr_matrix(adata.layers["counts"], dtype=np.float32)
adata.uns["expression_contract"] = {
    "x_scale": "raw_counts",
    "counts_layer": "counts",
    "counts_source": "X",
    "counts_validated": False,
    "counts_integer_check": None,
    "soupx_layer": None,
    "processing_history": [],
    "stage": "01",
}
print("expression_contract 已建立: x_scale=raw_counts, counts_validated=False")


## 样本级 QC 摘要

按 sample_id 分组统计每个样本的细胞数、基因中位数、UMI 中位数、线粒体比例中位数。
**看什么**：是否存在某个样本与其他样本差异过大（如某样本细胞数极少、或 MT% 异常偏高）。
这将帮助判断是否需要为特定样本设置差异化阈值。

In [ ]:
# 样本级 QC 摘要表
print("===== 样本级 QC 摘要 =====")
sample_summary = adata.obs.groupby("sample_id").agg(
    n_cells=("n_genes", "count"),
    median_n_genes=("n_genes", "median"),
    median_total_counts=("total_counts", "median"),
    median_pct_mt=("pct_counts_mt", "median"),
).sort_values("n_cells", ascending=False)
display(sample_summary)

abnormal = sample_summary[
    (sample_summary["n_cells"] < 100) | (sample_summary["median_pct_mt"] > 30)
]
if len(abnormal) > 0:
    print("\nWARNING 异常样本（n_cells<100 或 median_pct_mt>30%）：")
    display(abnormal)
else:
    print("\n所有样本通过初步检查。")


## 基线 QC 分布

绘制三个主 QC 指标的小提琴图和散点图，供 PI 在设定过滤阈值前直观判断数据质量。

**三个指标的含义**：
- `n_genes`：每个细胞检测到的基因数。过低→空液滴或死细胞；过高→可能是双细胞
- `total_counts`：每个细胞的总 UMI 计数。分布应与 n_genes 正相关
- `pct_counts_mt`：线粒体转录本百分比。过高（>20%）→细胞膜破损/凋亡

**如何使用这些图**：
1. 先看小提琴图，了解各指标的总体分布范围和离群情况
2. 再看散点图，检查 n_genes vs pct_mt 的关系——通常呈负相关
3. 根据分布特征，回到顶部 PARAMS 调整 N_MAD 或固定阈值

In [ ]:
# QC 小提琴图：按 sample_id 分组展示三个主指标。
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, metric in enumerate(["n_genes", "total_counts", "pct_counts_mt"]):
    ax = axes[i]
    sc.pl.violin(adata, keys=metric, groupby="sample_id", rotation=45, ax=ax, show=False)
    ax.set_title(f"{metric}（过滤前）")
plt.tight_layout()
fig.savefig("results/figures/01_kim_qc_violin_pre.png", dpi=150, bbox_inches="tight")
plt.show()

# QC 散点图
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc.pl.scatter(adata, x="total_counts", y="n_genes", color="pct_counts_mt", ax=axes[0], show=False)
axes[0].set_title("total_counts vs n_genes（按 pct_mt 着色，过滤前）")
sc.pl.scatter(adata, x="n_genes", y="pct_counts_mt", color="total_counts", ax=axes[1], show=False)
axes[1].set_title("n_genes vs pct_mt（按 total_counts 着色，过滤前）")
plt.tight_layout()
fig.savefig("results/figures/01_kim_qc_scatter_pre.png", dpi=150, bbox_inches="tight")
plt.show()


## 自适应阈值计算（MAD-based）

MAD（median absolute deviation）是比标准差更稳健的离散度度量，对离群值不敏感。

**参数含义**：
- `N_MAD = 3`：阈值 = 中位数 +/- N_MAD * MAD
- 越大越宽松（保留更多细胞），越小越严格（去除更多细胞）
- `n_genes` 做双侧过滤（过低+过高），`total_counts` 仅下界，`pct_counts_mt` 仅上界

**Why MAD 而不是固定阈值**：不同数据集/样本的 baseline 差异很大（如组织活检 vs 类器官的 MT% 基线差异可达 3-5 倍）。
MAD 基于每个数据集自身的分布自适应调整，避免用一个固定阈值削足适履。

In [ ]:
# 自适应阈值计算（MAD-based）
n_before = adata.n_obs

# 过滤前统计摘要
qc_pre_stats = {
    "n_genes": {"median": float(adata.obs["n_genes"].median()), "mean": float(adata.obs["n_genes"].mean())},
    "total_counts": {"median": float(adata.obs["total_counts"].median()), "mean": float(adata.obs["total_counts"].mean())},
    "pct_counts_mt": {"median": float(adata.obs["pct_counts_mt"].median()), "mean": float(adata.obs["pct_counts_mt"].mean())},
}

if QC_STRATEGY == "adaptive":
    thresholds = {}
    for metric, direction in [("n_genes", "both"), ("total_counts", "lower"), ("pct_counts_mt", "upper")]:
        if PER_SAMPLE_MAD:
            # 按 sample_id 分组，每个样本独立计算阈值
            sample_thresholds = {}
            for sample_id in adata.obs["sample_id"].unique():
                mask = adata.obs["sample_id"] == sample_id
                vals = adata.obs.loc[mask, metric].dropna()
                if len(vals) < 10:
                    print(f"  WARNING: {sample_id} 仅有 {len(vals)} 个细胞的 {metric} 值，沿用全局阈值")
                    continue
                med = vals.median()
                mad_val = median_abs_deviation(vals, nan_policy="omit")
                lower = max(0, med - N_MAD * mad_val) if direction in ("both", "lower") else None
                upper = med + N_MAD * mad_val if direction in ("both", "upper") else None
                sample_thresholds[sample_id] = {"median": round(med, 1), "mad": round(mad_val, 1),
                                                 "lower": round(lower, 1) if lower else None,
                                                 "upper": round(upper, 1) if upper else None}
            thresholds[metric] = {"mode": "per_sample", "per_sample": sample_thresholds}
        else:
            # 全局 MAD（原逻辑）
            vals = adata.obs[metric].dropna()
            med = vals.median()
            mad = median_abs_deviation(vals, nan_policy="omit")
            lower = max(0, med - N_MAD * mad) if direction in ("both", "lower") else None
            upper = med + N_MAD * mad if direction in ("both", "upper") else None
            thresholds[metric] = {"median": round(med, 1), "mad": round(mad, 1),
                                  "lower": round(lower, 1) if lower else None,
                                  "upper": round(upper, 1) if upper else None}
elif QC_STRATEGY == "fixed":
    # 固定阈值模式：适用于已知阈值的数据集（如类器官、实验室内部标准）
    thresholds = {
        "n_genes": {"median": None, "mad": None, "lower": MIN_GENES if 'MIN_GENES' in dir() else 200, "upper": MAX_GENES if 'MAX_GENES' in dir() else 6000},
        "total_counts": {"median": None, "mad": None, "lower": MIN_COUNTS if 'MIN_COUNTS' in dir() else 500, "upper": None},
        "pct_counts_mt": {"median": None, "mad": None, "lower": None, "upper": MAX_PCT_MT if 'MAX_PCT_MT' in dir() else 20},
    }
    print("使用固定阈值模式")
else:
    raise ValueError(f"不支持的 QC_STRATEGY: {QC_STRATEGY}，请使用 'adaptive' 或 'fixed'")

print(f"===== QC 阈值（{QC_STRATEGY}）=====")
print(f"  策略: {QC_STRATEGY}")
if QC_STRATEGY == "adaptive":
    print(f"  N_MAD: {N_MAD}")
    if PER_SAMPLE_MAD:
        print(f"  模式: per-sample（每个 sample_id 独立计算 MAD）")
    else:
        print(f"  模式: global（全局 MAD）")

if PER_SAMPLE_MAD:
    # 打印每个 sample 的阈值汇总表
    print("\n===== 各样本阈值明细 =====")
    for metric in ["n_genes", "total_counts", "pct_counts_mt"]:
        print(f"\n--- {metric} ---")
        rows = []
        for sid, t in thresholds[metric]["per_sample"].items():
            rows.append({"sample_id": sid, **{k: v for k, v in t.items() if v is not None}})
        if rows:
            display(pd.DataFrame(rows).set_index("sample_id"))
else:
    thresh_df = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if vv is not None} for k, v in thresholds.items()}).T
    display(thresh_df)

In [ ]:
# --- 跨样本阈值 forest plot（仅 per-sample 模式）---
# 展示各样本的 MAD 阈值范围，可直观比较各样本的 QC 特性差异
if PER_SAMPLE_MAD and QC_STRATEGY == "adaptive":
    fig, axes = plt.subplots(1, 3, figsize=(15, max(4, len(adata.obs["sample_id"].unique()) * 0.4)))
    for i, metric in enumerate(["n_genes", "total_counts", "pct_counts_mt"]):
        ax = axes[i]
        per_sample = thresholds[metric].get("per_sample", {})
        samples = sorted(per_sample.keys())
        if not samples:
            ax.set_title(f"{metric}\n（无 per-sample 数据）")
            continue
        y_pos = range(len(samples))
        lowers = [per_sample[s].get("lower", 0) or 0 for s in samples]
        uppers = [per_sample[s].get("upper", 0) or 0 for s in samples]
        medians = [per_sample[s].get("median", 0) for s in samples]
        ax.barh(y_pos, [u - l for u, l in zip(uppers, lowers)], left=lowers, height=0.6,
                alpha=0.3, color="steelblue")
        ax.scatter(medians, y_pos, color="red", zorder=5, s=20, label="median")
        ax.set_yticks(y_pos)
        ax.set_yticklabels(samples, fontsize=8)
        ax.set_xlabel(metric)
        ax.set_title(f"{metric} 阈值范围")
        if i == 0:
            ax.legend(fontsize=7, loc="lower right")
    plt.suptitle("Per-sample MAD 阈值 Forest Plot", fontsize=12)
    plt.tight_layout()
    plt.savefig("results/figures/01_kim_qc_forest.png", dpi=150, bbox_inches="tight")
    plt.show()
elif PER_SAMPLE_MAD and QC_STRATEGY == "fixed":
    print("fixed 模式下无 per-sample 阈值，跳过 forest plot")


## N_MAD 敏感度分析

核心问题：N_MAD 太小 -> 丢太多细胞（可能丢真信号）；太大 -> 保留垃圾。
经验法则：组织活检 3-4，类器官 5-7。下面的曲线帮助你选择最佳值。


In [ ]:
# === N_MAD 敏感度分析：帮助 PI 选择最优阈值 ===
# F4修复：敏感度曲线与过滤同口径——PER_SAMPLE_MAD 开关同时作用于曲线与过滤
# 此前曲线始终使用全局 MAD，PI 据曲线选参数会被误导
_test_mads = [3, 4, 5, 6, 7]
sensitivity_results = []
for _nm in _test_mads:
    _keep = pd.Series(True, index=adata.obs_names)
    for metric, direction in [("n_genes", "both"), ("total_counts", "lower"), ("pct_counts_mt", "upper")]:
        if PER_SAMPLE_MAD:
            # 按 sample_id 分组独立算 MAD（与阈值计算 + 实际过滤同口径）
            for sample_id in adata.obs["sample_id"].unique():
                smask = adata.obs["sample_id"] == sample_id
                vals = adata.obs.loc[smask, metric].dropna()
                # 样本细胞数不足 10，无法可靠估计 MAD，该样本细胞不参与本维度过滤
                if len(vals) < 10:
                    continue
                med = vals.median()
                mad_val = median_abs_deviation(vals, nan_policy="omit")
                if direction in ("both", "lower"):
                    _keep.loc[smask] &= adata.obs.loc[smask, metric] >= max(0, med - _nm * mad_val)
                if direction in ("both", "upper"):
                    _keep.loc[smask] &= adata.obs.loc[smask, metric] <= med + _nm * mad_val
        else:
            vals = adata.obs[metric].dropna()
            med = vals.median()
            mad_val = median_abs_deviation(vals, nan_policy="omit")
            if direction in ("both", "lower"):
                _keep &= adata.obs[metric] >= max(0, med - _nm * mad_val)
            if direction in ("both", "upper"):
                _keep &= adata.obs[metric] <= med + _nm * mad_val
    n_keep = _keep.sum()
    sensitivity_results.append({
        "N_MAD": _nm,
        "cells_kept": n_keep,
        "pct_kept": round(100 * n_keep / adata.n_obs, 1),
        "median_mt_kept": round(adata.obs.loc[_keep, "pct_counts_mt"].median(), 2),
    })

sens_df = pd.DataFrame(sensitivity_results)
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(sens_df["N_MAD"], sens_df["pct_kept"], "o-", color="steelblue", linewidth=2)
ax1.axvline(N_MAD, color="red", linestyle="--", label=f"当前 N_MAD={N_MAD}")
ax1.set_xlabel("N_MAD")
ax1.set_ylabel("细胞保留率 (%)", color="steelblue")
ax1.set_title("N_MAD 敏感度：保留率 vs 阈值宽松度")
ax2 = ax1.twinx()
ax2.plot(sens_df["N_MAD"], sens_df["median_mt_kept"], "s--", color="orange")
ax2.set_ylabel("保留细胞的 median MT%", color="orange")
ax1.legend()
plt.tight_layout()
plt.savefig("results/figures/01_kim_mad_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()
print(sens_df.to_string(index=False))
print("\n经验判据：保留 85-95% 细胞 + median MT% 不显著上升 = 合理")


## 基因复杂度

`log_complexity = log10(n_genes+1) / log10(total_counts+1)` 反映每个细胞的"基因多样性密度"。
复杂度异常低（相同 UMI 下基因数过少）提示该细胞可能只捕获了极少数高表达基因，
是低质量细胞的补充判据。

In [ ]:
# 基因复杂度
adata.obs["log_complexity"] = np.log10(adata.obs["n_genes"] + 1) / np.log10(adata.obs["total_counts"] + 1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(adata.obs["log_complexity"].dropna(), bins=50, color="steelblue", edgecolor="white")
ax.set_xlabel("log10(n_genes+1) / log10(total_counts+1)")
ax.set_ylabel("细胞数")
ax.set_title("基因复杂度分布")
for pct, color in [(1, "red"), (5, "orange"), (50, "green"), (95, "orange"), (99, "red")]:
    val = np.percentile(adata.obs["log_complexity"].dropna(), pct)
    ax.axvline(val, color=color, linestyle="--", alpha=0.5, linewidth=0.8)
plt.tight_layout()
fig.savefig("results/figures/01_kim_complexity.png", dpi=150, bbox_inches="tight")
plt.show()
for pct in [1, 5, 25, 50, 75, 95, 99]:
    print(f"  P{pct}: {np.percentile(adata.obs['log_complexity'].dropna(), pct):.4f}")


## 特殊基因标记

**血红蛋白基因（HB）**：红细胞污染/溶解的标志。仅标记不移除——消化道活检中低水平 HB 背景常见。
**应激基因**：即时早期基因（IEGs）+ 热休克蛋白（HSPs），在组织解离过程中被机械/酶切应激诱导。
标记供下游分析参考，不是过滤依据。

In [ ]:
# FLAG_HEMOGLOBIN=False，跳过血红蛋白标记（类器官数据集无 RBC 污染）。

# 应激基因标记（仅标记，不移除）
if FLAG_STRESS_GENES:
    STRESS_GENES = ['JUN', 'FOS', 'JUNB', 'FOSB', 'ATF3', 'HSPA1A', 'HSPA1B', 'HSP90AA1', 'HSP90AB1', 'DNAJB1', 'HSPB1']
    stress_in_data = [g for g in STRESS_GENES if g in adata.var_names]
    if stress_in_data:
        adata.var["stress"] = adata.var_names.isin(stress_in_data)
        sc.pp.calculate_qc_metrics(adata, qc_vars=["stress"], percent_top=None, log1p=False, inplace=True)
        print(f"应激基因: {len(stress_in_data)}/{len(STRESS_GENES)} 个检测到, 中位 pct={adata.obs['pct_counts_stress'].median():.2f}%")
    else:
        print("WARNING: 数据中未检测到应激基因")
else:
    print("⏭ 跳过应激基因标记（FLAG_STRESS_GENES=False）")

## 双细胞鉴定（Scrublet）

双细胞（doublet）是两个细胞被误包在同一个液滴中测序。其基因表达是两种细胞类型的混合，
会干扰细胞类型注释和差异表达分析。

**处理策略**：使用 manifest-driven 跳过逻辑。
- 若 manifest `preprocessing_done` 含 `doublet_removal`→跳过（作者已处理）
- 若 manifest `qc_overrides.doublet_removal.skip=true`→跳过
- 否则按样本独立运行 scrublet

**注意**：doublet 仅标记不移除。PI 在查看下游聚类后可根据 doublet 是否形成独立小群来决定。

In [ ]:
# 双细胞鉴定（决策8）：三态定级 + per-sample scrublet
# 
# 本节只做检测执行与 per-sample 三态定级（singlet/uncertain/doublet）。
# include 派生、needs_review 诊断、阈值持久化由后续 cell 完成。
import scrublet as scr
# scrublet 版本号（用于 provenance 记录）
try:
    _scrublet_version = scr.__version__
except AttributeError:
    try:
        from importlib.metadata import version
        _scrublet_version = version("scrublet")
    except Exception:
        _scrublet_version = "unknown"

# ---- obs 列初始化（fresh-kernel 安全，不依赖残留状态）----
# 未检测样本/跳过场景所有列保留默认值，确保下游 02/测试不因列缺失报错
adata.obs["doublet_score"] = np.nan          # float；NaN 表示未检测
adata.obs["doublet_class"] = "singlet"       # 分类：singlet | uncertain | doublet；默认 singlet
adata.obs["predicted_doublet"] = False       # 向后兼容旧列：= doublet_class=="doublet"（仅高置信）
adata.obs["doublet_include"] = True          # 下游纳入控制；默认 True（只排除高置信 doublet）

# ---- manifest-driven skip check（保留既有逻辑）----
with open(MANIFEST_PATH) as f:
    manifest = yaml.safe_load(f)
pp_done = manifest.get("preprocessing_done", [])
qc_override = manifest.get("qc_overrides", {}).get("doublet_removal", {})

skip_doublet = False
skip_reason = None
if "doublet_removal" in pp_done:
    skip_doublet = True
    skip_reason = "原作者已去除双细胞（preprocessing_done 含 doublet_removal）"
elif qc_override.get("skip"):
    skip_doublet = True
    skip_reason = qc_override.get("reason", "qc_overrides.doublet_removal.skip=True")

# ---- per-sample 诊断容器 + 阈值记录 ----
n_doublets_total = 0  # 初始化（避免 EXPECTED_DOUBLET_RATE=None 时 NameError）
doublet_diagnostics = {}         # per-sample 诊断（写入 uns["doublet_contract"]）
per_sample_thresholds = {}       # per-sample 阈值记录
doublet_needs_review = False     # 全局 needs_review 布尔（供 checkpoint cell 消费）
needs_review_reasons = []        # needs_review 原因列表

if skip_doublet:
    # 跳过检测：所有细胞保持默认 singlet/include=True
    print(f"双细胞鉴定已跳过: {skip_reason}")
    doublet_method = "skipped"
    doublet_method_version = "n/a"
elif EXPECTED_DOUBLET_RATE is None:
    # 未设定预期率 → 不运行 scrublet
    print("EXPECTED_DOUBLET_RATE=None，未运行 scrublet（所有细胞标记 singlet，include 全 True）")
    doublet_method = "not_run"
    doublet_method_version = "n/a"
else:
    # 运行 Scrublet per sample
    doublet_method = "scrublet"
    doublet_method_version = _scrublet_version
    print(f"运行 Scrublet per sample（expected_doublet_rate={EXPECTED_DOUBLET_RATE}）...")
    
    for sample_id in sorted(adata.obs["sample_id"].unique()):
        sample_mask = adata.obs["sample_id"] == sample_id
        n_sample = sample_mask.sum()
        
        # ---- 细胞数不足 → 记 needs_review，不静默跳过 ----
        if n_sample < DOUBLET_MIN_CELLS:
            reason = f"too_few_cells: {sample_id} ({n_sample} cells < {DOUBLET_MIN_CELLS})"
            needs_review_reasons.append(reason)
            doublet_needs_review = True
            print(f"  {sample_id}: {n_sample} 细胞（< {DOUBLET_MIN_CELLS}），记 needs_review，"
                  f"该样本全部标记 singlet")
            doublet_diagnostics[sample_id] = {
                "n_cells": n_sample, "n_singlet": n_sample, "n_uncertain": 0,
                "n_doublet": 0, "doublet_rate": 0.0, "uncertain_rate": 0.0,
                "threshold_low": None, "threshold_high": None,
                "auto_threshold": None,
                "score_p50": None, "score_p90": None, "score_p99": None,
                "needs_review": True, "reason": reason,
            }
            per_sample_thresholds[sample_id] = {"low": None, "high": None, "auto": None}
            continue
        
        # ---- 子集 copy + scrublet 检测（内存纪律：子集 .copy() 为既有模式）----
        sub = adata[sample_mask].copy()
        scrub = scr.Scrublet(
            sub.X, expected_doublet_rate=EXPECTED_DOUBLET_RATE,
            random_state=RANDOM_SEED,
        )
        doublet_scores, _ = scrub.scrub_doublets()
        auto_threshold = scrub.threshold_
        
        # ---- 定阈值：high 优先用 DOUBLET_SCORE_HIGH，其次 DOUBLET_SCORE_THRESHOLD（旧手动），最后自动 ----
        # 向后兼容：DOUBLET_SCORE_THRESHOLD 非 None 时覆盖 high（并打印说明）
        if DOUBLET_SCORE_THRESHOLD is not None:
            threshold_high = DOUBLET_SCORE_THRESHOLD
            print(f"  使用手动 DOUBLET_SCORE_THRESHOLD={threshold_high}（覆盖自动阈值）")
        elif DOUBLET_SCORE_HIGH is not None:
            threshold_high = DOUBLET_SCORE_HIGH
        else:
            threshold_high = auto_threshold if auto_threshold is not None else 0.25
            if auto_threshold is None:
                _reason = f"scrublet_threshold_none: {sample_id}（auto_threshold=None，fallback 0.25）"
                needs_review_reasons.append(_reason)
                doublet_needs_review = True
        
        # low 优先用 DOUBLET_SCORE_LOW，否则 high*0.5 派生
        threshold_low = DOUBLET_SCORE_LOW if DOUBLET_SCORE_LOW is not None else threshold_high * 0.5
        
        # ---- 向量化三态定级（不逐行循环）----
        classes = np.full(n_sample, "singlet", dtype=object)
        classes[doublet_scores > threshold_high] = "doublet"
        classes[(doublet_scores >= threshold_low) & (doublet_scores <= threshold_high)] = "uncertain"
        
        # 写回 adata.obs
        adata.obs.loc[sample_mask, "doublet_score"] = doublet_scores
        adata.obs.loc[sample_mask, "doublet_class"] = classes
        
        # ---- 直方图可视化（三区标注：绿=singlet界，红=doublet界）----
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(doublet_scores, bins=50, color="steelblue", edgecolor="white", alpha=0.7)
        ax.axvline(threshold_low, color="green", linestyle=":", linewidth=1.5,
                   label=f"singlet/uncertain={threshold_low:.3f}")
        ax.axvline(threshold_high, color="red", linestyle="--", linewidth=1.5,
                   label=f"uncertain/doublet={threshold_high:.3f}")
        if auto_threshold is not None and abs(auto_threshold - threshold_high) > 1e-6:
            ax.axvline(auto_threshold, color="orange", linestyle="-.", linewidth=1,
                       label=f"auto={auto_threshold:.3f}")
        ax.set_xlabel("Doublet Score")
        ax.set_ylabel("细胞数")
        ax.set_title(f"{sample_id}: Doublet Score 分布（singlet | uncertain | doublet）")
        ax.legend(fontsize=8)
        plt.tight_layout(); plt.show()
        
        # ---- per-sample 诊断 ----
        n_dbl = int((classes == "doublet").sum())
        n_unc = int((classes == "uncertain").sum())
        n_sgl = int((classes == "singlet").sum())
        d_rate = n_dbl / n_sample
        u_rate = n_unc / n_sample
        
        score_p50 = round(float(np.percentile(doublet_scores, 50)), 4)
        score_p90 = round(float(np.percentile(doublet_scores, 90)), 4)
        score_p99 = round(float(np.percentile(doublet_scores, 99)), 4)
        
        # 样本级 needs_review 判定
        sample_needs_review = False
        sample_reasons = []
        if d_rate > DOUBLET_RATE_ALERT_HIGH:
            sample_needs_review = True
            sample_reasons.append(f"doublet_rate {d_rate:.1%} > alert {DOUBLET_RATE_ALERT_HIGH:.1%}")
        if DOUBLET_RATE_ALERT_LOW is not None and d_rate < DOUBLET_RATE_ALERT_LOW:
            sample_needs_review = True
            sample_reasons.append(f"doublet_rate {d_rate:.1%} < alert_low {DOUBLET_RATE_ALERT_LOW:.1%}")
        if sample_needs_review:
            doublet_needs_review = True
            needs_review_reasons.extend(
                [f"{sample_id}: {r}" for r in sample_reasons]
            )
        
        doublet_diagnostics[sample_id] = {
            "n_cells": n_sample, "n_singlet": n_sgl, "n_uncertain": n_unc,
            "n_doublet": n_dbl,
            "doublet_rate": round(d_rate, 4),
            "uncertain_rate": round(u_rate, 4),
            "threshold_low": round(threshold_low, 4),
            "threshold_high": round(threshold_high, 4),
            "auto_threshold": round(auto_threshold, 4) if auto_threshold is not None else None,
            "score_p50": score_p50, "score_p90": score_p90, "score_p99": score_p99,
            "needs_review": sample_needs_review,
            "reason": "; ".join(sample_reasons) if sample_reasons else None,
        }
        per_sample_thresholds[sample_id] = {
            "low": round(threshold_low, 4),
            "high": round(threshold_high, 4),
            "auto": round(auto_threshold, 4) if auto_threshold is not None else None,
        }
        
        status_flag = " [NEEDS_REVIEW]" if sample_needs_review else ""
        print(f"  {sample_id}: sgl={n_sgl} unc={n_unc} dbl={n_dbl} / {n_sample} "
              f"(dbl_rate={d_rate:.1%}, unc_rate={u_rate:.1%}){status_flag}")

print(f"\n检测完成：method={doublet_method}，"
      f"全局 needs_review={'是' if doublet_needs_review else '否'}")

In [ ]:
# 双细胞三态收尾：include 列派生 + needs_review 诊断 + 持久化到 uns["doublet_contract"]
# 
# 决策8：只标记不物理删除细胞（01 保存完整对象；02 按 doublet_include 构建整合对象）

# ---- include 列派生（决策8：排除高置信 doublet；uncertain 由 DOUBLET_UNCERTAIN_INCLUDE 决定）----
# 重置所有为 True，再按规则排除
adata.obs["doublet_include"] = True
adata.obs.loc[adata.obs["doublet_class"] == "doublet", "doublet_include"] = False
if not DOUBLET_UNCERTAIN_INCLUDE:
    adata.obs.loc[adata.obs["doublet_class"] == "uncertain", "doublet_include"] = False

# 向后兼容：predicted_doublet = 仅高置信 doublet（旧列语义不变）
adata.obs["predicted_doublet"] = adata.obs["doublet_class"] == "doublet"

# ---- 全局汇总 ----
n_dbl = int((adata.obs["doublet_class"] == "doublet").sum())
n_unc = int((adata.obs["doublet_class"] == "uncertain").sum())
n_sgl = int((adata.obs["doublet_class"] == "singlet").sum())
n_excluded = int((~adata.obs["doublet_include"]).sum())

print(f"双细胞三态汇总: singlet={n_sgl}  uncertain={n_unc}  doublet={n_dbl}")
print(f"  纳入下游={adata.n_obs - n_excluded}  排除={n_excluded}"
      f"（其中 high-conf doublet={n_dbl}"
      + (f", uncertain 排除={n_unc}" if not DOUBLET_UNCERTAIN_INCLUDE else "")
      + "）")

if doublet_needs_review:
    print(f"\n[NEEDS_REVIEW] 双细胞检测存在异常，需 PI 审阅 per-sample 诊断表：")
    for r in needs_review_reasons:
        print(f"  - {r}")
else:
    print("\n双细胞检测无异常。")

# ---- 每样本诊断表打印 ----
print("\n--- Per-sample 双细胞诊断 ---")
for sid in sorted(doublet_diagnostics.keys()):
    d = doublet_diagnostics[sid]
    nr_flag = " [NEEDS_REVIEW]" if d.get("needs_review") else ""
    print(f"  {sid}: {d['n_cells']} cells | "
          f"sgl={d['n_singlet']} unc={d['n_uncertain']} dbl={d['n_doublet']} | "
          f"dbl_rate={d['doublet_rate']:.1%} unc_rate={d['uncertain_rate']:.1%} | "
          f"thr=[{d['threshold_low']}, {d['threshold_high']}]{nr_flag}")
    if d.get("reason"):
        print(f"        reason: {d['reason']}")

# ---- 阈值与元数据持久化到 uns["doublet_contract"]（独立于 expression_contract）----
adata.uns["doublet_contract"] = {
    "method": doublet_method,
    "method_version": doublet_method_version,
    "per_sample_thresholds": per_sample_thresholds,
    "uncertain_include": DOUBLET_UNCERTAIN_INCLUDE,
    "rate_alert_high": DOUBLET_RATE_ALERT_HIGH,
    "rate_alert_low": DOUBLET_RATE_ALERT_LOW,
    "n_singlet": n_sgl,
    "n_uncertain": n_unc,
    "n_doublet": n_dbl,
    "n_excluded": n_excluded,
    "needs_review": bool(doublet_needs_review),
    "needs_review_reasons": needs_review_reasons,
    "per_sample_diagnostics": doublet_diagnostics,
    "random_seed": RANDOM_SEED,
}
print(f"\ndoublet_contract 已写入 uns（method={doublet_method}，"
      f"needs_review={bool(doublet_needs_review)}）")

## 环境 RNA 校正（SoupX）

环境 RNA（ambient RNA）来自裂解的细胞碎片和游离 RNA，悬浮在液滴溶液中并被随机捕获形成背景噪声。
SoupX 利用 raw matrix（含空液滴/碎片背景）和 filtered matrix（只含真实细胞）之间的差异，
估计每个基因的污染比例并扣除。

**为什么用 subprocess Rscript 而不是 rpy2**：
rpy2 + anndata2ri 在 conda R 4.4.3 下存在严重兼容性问题。subprocess 独立进程通过临时 mtx 文件交换数据，进程隔离避免桥接崩溃。

**三重守卫**（任一不满足则优雅跳过）：
1. `SOUPX_ENABLED=True`
2. Manifest 声明了 `raw_matrix_path`
3. Rscript 可执行 + SoupX R 包可加载

In [ ]:
# 环境 RNA 校正（SoupX）—— subprocess Rscript 模式
soupx_applied = False
n_soupx_corrected = 0

# 初始化 counts_soupx layer（从原始 counts 拷贝作基线）
if "counts_soupx" not in adata.layers:
    adata.layers["counts_soupx"] = adata.layers["counts"].copy()
    if not sp.issparse(adata.layers["counts_soupx"]) or adata.layers["counts_soupx"].dtype != np.float32:
        adata.layers["counts_soupx"] = sp.csr_matrix(adata.layers["counts_soupx"], dtype=np.float32)
_counts_checksum = adata.layers["counts"].sum()

if not SOUPX_ENABLED:
    print(f"SoupX 已跳过: SOUPX_ENABLED=False（类器官数据集无 raw matrix）。")
elif not adata.uns.get("raw_matrix_path"):
    print("SoupX 已跳过: adata.uns 中无 raw_matrix_path。")
elif not R_AVAILABLE:
    print("SoupX 已跳过: R 环境未就绪。")
else:
    raw_path = adata.uns["raw_matrix_path"]
    soupx_script = "scripts/soupx_run.R"
    soupx_tmp = "results/_soupx_tmp"
    os.makedirs(soupx_tmp, exist_ok=True)
    import scipy.io

    print(f"raw_matrix_path: {raw_path}")
    if "ambient_correction_applied" not in adata.obs.columns:
        adata.obs["ambient_correction_applied"] = False

    for sample_id in sorted(adata.obs["sample_id"].unique()):
        sample_mask = adata.obs["sample_id"] == sample_id
        n_cells = sample_mask.sum()
        cell_ids = adata.obs_names[sample_mask]

        # 提取原始 10x barcode
        prefix = f"{sample_id}_"
        original_barcodes = []
        for cid in cell_ids:
            if not cid.startswith(prefix):
                warnings.warn(f"cell_id '{cid}' barcode 解析异常")
                original_barcodes.append(cid)
                continue
            rest = cid[len(prefix):]
            parts = rest.rsplit("-", 1)
            original_barcodes.append(parts[0] if len(parts) == 2 and parts[1].isdigit() else rest)

        # 定位 raw matrix
        raw_sample_dir = Path(raw_path) / sample_id / "raw_feature_bc_matrix"
        if not (raw_sample_dir / "matrix.mtx.gz").exists() and not (raw_sample_dir / "matrix.mtx").exists():
            alt_raw = Path(raw_path) / "raw_feature_bc_matrix"
            if (alt_raw / "matrix.mtx.gz").exists() or (alt_raw / "matrix.mtx").exists():
                raw_sample_dir = alt_raw
            else:
                print(f"  {sample_id}: raw_feature_bc_matrix 未找到，跳过")
                continue

        try:
            sub_adata = adata[cell_ids].copy()
            filtered_export = os.path.join(soupx_tmp, f"{sample_id}_filtered")
            os.makedirs(filtered_export, exist_ok=True)
            count_mtx = sp.csr_matrix(sub_adata.X).T
            scipy.io.mmwrite(os.path.join(filtered_export, "matrix.mtx"), count_mtx)
            with open(os.path.join(filtered_export, "barcodes.tsv"), "w") as f:
                f.write("\n".join(original_barcodes) + "\n")
            with open(os.path.join(filtered_export, "features.tsv"), "w") as f:
                for gn in sub_adata.var_names:
                    f.write(f"{gn}\t{gn}\tGene Expression\n")

            work_dir = os.path.join(soupx_tmp, sample_id)
            cmd = [RSCRIPT_BIN, "--vanilla", soupx_script,
                   os.path.abspath(work_dir), os.path.abspath(filtered_export),
                   os.path.abspath(str(raw_sample_dir)), sample_id]
            print(f"  {sample_id}: 执行 SoupX...")
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
            for line in result.stdout.strip().split("\n"):
                print(f"    [R] {line}")
            if result.returncode != 0:
                print(f"  Rscript 失败 (exit={result.returncode}): {result.stderr[:300]}")
                continue

            corrected_mtx_fp = os.path.join(work_dir, "corrected_counts.mtx")
            if not os.path.exists(corrected_mtx_fp):
                print(f"  校正矩阵未产出: {corrected_mtx_fp}")
                continue
            corrected = sp.csr_matrix(scipy.io.mmread(corrected_mtx_fp).T)
            if corrected.shape != (n_cells, sub_adata.n_vars):
                print(f"  形状不匹配，跳过")
                continue

            # barcode + gene order validation
            with open(os.path.join(work_dir, "barcodes.tsv")) as f:
                if [l.strip() for l in f] != original_barcodes:
                    print(f"  barcode 顺序不匹配，跳过")
                    continue
            out_features = pd.read_csv(os.path.join(work_dir, "features.tsv"), sep="\t", header=None)
            if out_features.iloc[:, 1].tolist() != sub_adata.var_names.tolist():
                print(f"  基因顺序不匹配，跳过")
                continue

            # F1修复：对布尔/花式索引产生的 AnnData 视图赋值不会写回原对象
            # scanpy 只弹 warning 而不实际修改底层 CSR 矩阵中的值
            # 改为整数位置索引直接写 CSR 矩阵，并做校正前后差异断言防止静默失败
            cell_indices = np.where(sample_mask.values)[0]
            adata.layers["counts_soupx"][cell_indices, :] = corrected
            # counts checksum 守卫：确保原始 counts layer 未被修改
            _new_checksum = adata.layers["counts"].sum()
            assert abs(_new_checksum - _counts_checksum) < 1, (
                f"counts layer checksum changed: {_counts_checksum} -> {_new_checksum}"
            )
            adata.obs.loc[cell_ids, "ambient_correction_applied"] = True
            n_soupx_corrected += n_cells
            print(f"  {sample_id}: SoupX 完成, {n_cells} 细胞已校正 -> layers['counts_soupx']")
            del sub_adata
        except subprocess.TimeoutExpired:
            print(f"  {sample_id}: 超时（10min），跳过")
        except Exception as e:
            print(f"  {sample_id}: 异常 ({type(e).__name__}): {e}")

    if n_soupx_corrected > 0:
        soupx_applied = True
    print(f"\nSoupX 校正: {n_soupx_corrected} 个细胞已校正")
    print(f"ambient_correction_applied: {adata.obs['ambient_correction_applied'].value_counts().to_dict()}")
    shutil.rmtree(soupx_tmp, ignore_errors=True)


## 细胞周期评分

使用 Tirosh et al. (2015) 的 S 期和 G2M 期 marker genes 对每个细胞打分。
细胞周期阶段（G1/S/G2M）是重要的技术协变量——如果不同样本/条件的细胞周期分布
不均衡，可能在差异表达分析中引入混淆。

In [ ]:
# 细胞周期评分（Tirosh 2015 marker genes）
if SCORE_CELL_CYCLE:
    s_genes = ["MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG","GINS2","MCM6","CDCA7","DTL","PRIM1","UHRF1","MLF1IP","HELLS","RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1","TIPIN","DSCC1","BLM","CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8"]
    g2m_genes = ["HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF","TACC3","FAM64A","SMC4","CCNB2","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP","CDCA3","HN1","CDC20","TTK","CDC25C","KIF2C","RANGAP1","NCAPD2","DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA","PSRC1","ANLN","LBR","CKAP5","CENPE","CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA"]
    sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)
    print("细胞周期评分完成")
    print(adata.obs["phase"].value_counts())
else:
    print("SCORE_CELL_CYCLE=False，跳过")


## 过滤

根据上一步得出的阈值过滤低质量细胞。

**不在此步移除的**：
- 双细胞（predicted_doublet）：仅标记，供下游聚类后决定
- 血红蛋白高表达细胞：仅标记（flag_hb）
- 应激高表达细胞：仅标记

In [ ]:
# 过滤：应用 QC 阈值
print("===== QC 过滤 =====")
cells_before = adata.n_obs
print(f"过滤前细胞数: {cells_before:,}")

if PER_SAMPLE_MAD:
    # Per-sample 模式：按每个 sample 的阈值独立判断
    keep = pd.Series(True, index=adata.obs_names)
    filter_counts = {"n_genes_lower": 0, "n_genes_upper": 0, "total_counts": 0, "pct_counts_mt": 0}
    for sample_id in adata.obs["sample_id"].unique():
        smask = adata.obs["sample_id"] == sample_id
        st = thresholds["n_genes"]["per_sample"].get(sample_id)
        if st and st.get("lower") is not None:
            n_fail = ((adata.obs.loc[smask, "n_genes"] < st["lower"]) & keep.loc[smask]).sum()
            keep.loc[smask] &= adata.obs.loc[smask, "n_genes"] >= st["lower"]
            filter_counts["n_genes_lower"] += n_fail
        if st and st.get("upper") is not None:
            n_fail = ((adata.obs.loc[smask, "n_genes"] > st["upper"]) & keep.loc[smask]).sum()
            keep.loc[smask] &= adata.obs.loc[smask, "n_genes"] <= st["upper"]
            filter_counts["n_genes_upper"] += n_fail
        st_tc = thresholds["total_counts"]["per_sample"].get(sample_id)
        if st_tc and st_tc.get("lower") is not None:
            n_fail = ((adata.obs.loc[smask, "total_counts"] < st_tc["lower"]) & keep.loc[smask]).sum()
            keep.loc[smask] &= adata.obs.loc[smask, "total_counts"] >= st_tc["lower"]
            filter_counts["total_counts"] += n_fail
        st_mt = thresholds["pct_counts_mt"]["per_sample"].get(sample_id)
        if st_mt and st_mt.get("upper") is not None:
            n_fail = ((adata.obs.loc[smask, "pct_counts_mt"] > st_mt["upper"]) & keep.loc[smask]).sum()
            keep.loc[smask] &= adata.obs.loc[smask, "pct_counts_mt"] <= st_mt["upper"]
            filter_counts["pct_counts_mt"] += n_fail
    print(f"  n_genes 偏低: {filter_counts['n_genes_lower']:,}  偏高: {filter_counts['n_genes_upper']:,}")
    print(f"  total_counts 偏低: {filter_counts['total_counts']:,}")
    print(f"  pct_counts_mt 偏高: {filter_counts['pct_counts_mt']:,}")
else:
    keep = pd.Series(True, index=adata.obs_names)
    if thresholds["n_genes"]["lower"] is not None:
        n_fail = (adata.obs["n_genes"] < thresholds["n_genes"]["lower"]).sum()
        keep &= adata.obs["n_genes"] >= thresholds["n_genes"]["lower"]
        print(f"  n_genes < {thresholds['n_genes']['lower']:.0f}: {n_fail:,} 失败")
    if thresholds["n_genes"]["upper"] is not None:
        n_fail = (adata.obs["n_genes"] > thresholds["n_genes"]["upper"]).sum()
        keep &= adata.obs["n_genes"] <= thresholds["n_genes"]["upper"]
        print(f"  n_genes > {thresholds['n_genes']['upper']:.0f}: {n_fail:,} 失败")
    if thresholds["total_counts"]["lower"] is not None:
        n_fail = (adata.obs["total_counts"] < thresholds["total_counts"]["lower"]).sum()
        keep &= adata.obs["total_counts"] >= thresholds["total_counts"]["lower"]
        print(f"  total_counts < {thresholds['total_counts']['lower']:.0f}: {n_fail:,} 失败")
    if thresholds["pct_counts_mt"]["upper"] is not None:
        n_fail = (adata.obs["pct_counts_mt"] > thresholds["pct_counts_mt"]["upper"]).sum()
        keep &= adata.obs["pct_counts_mt"] <= thresholds["pct_counts_mt"]["upper"]
        print(f"  pct_counts_mt > {thresholds['pct_counts_mt']['upper']:.1f}%: {n_fail:,} 失败")

print(f"\n保留细胞: {keep.sum():,} / {cells_before:,} ({100*keep.sum()/cells_before:.1f}%)")

obs_pre_filter = adata.obs.copy()  # 内存安全：只复制 obs DataFrame，不复制矩阵
adata = adata[keep].copy()
cells_after = adata.n_obs
print(f"去除细胞数: {cells_before - cells_after:,} ({100*(cells_before-cells_after)/cells_before:.1f}%)")

# 基因过滤：移除仅在极少细胞中检测到的噪声基因
n_genes_before = adata.n_vars
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
n_genes_after = adata.n_vars
print(f"基因过滤：{n_genes_before:,} → {n_genes_after:,}（移除 {n_genes_before - n_genes_after:,} 个仅在 <{MIN_CELLS_PER_GENE} 细胞中检测到的基因）")

## 过滤交叉诊断

MAD 过滤与各标记物（doublet/HB/stress）的关系：被过滤掉的细胞是否富集了这些标记？


In [ ]:
# === 过滤交叉诊断：MAD 过滤与各标记物的关系 ===
n_removed_total = cells_before - cells_after
if n_removed_total > 0 and 'obs_pre_filter' in dir():
    print("===== 被过滤细胞的标记物富集分析 =====")
    _removed_idx = obs_pre_filter.index.difference(adata.obs_names)
    _kept_idx = adata.obs_names
    
    enrichment = {}
    if "predicted_doublet" in obs_pre_filter.columns:
        dbl_removed = obs_pre_filter.loc[_removed_idx, "predicted_doublet"].mean()
        dbl_kept = obs_pre_filter.loc[_kept_idx, "predicted_doublet"].mean()
        enrichment["doublet"] = {
            "在被过滤细胞中": f"{dbl_removed:.1%}",
            "在保留细胞中": f"{dbl_kept:.1%}",
            "富集倍数": round(dbl_removed / max(dbl_kept, 0.001), 1)
        }
    if "flag_hb" in obs_pre_filter.columns:
        hb_removed = obs_pre_filter.loc[_removed_idx, "flag_hb"].mean()
        hb_kept = obs_pre_filter.loc[_kept_idx, "flag_hb"].mean()
        enrichment["hemoglobin"] = {
            "在被过滤细胞中": f"{hb_removed:.1%}",
            "在保留细胞中": f"{hb_kept:.1%}",
            "富集倍数": round(hb_removed / max(hb_kept, 0.001), 1)
        }
    if "pct_counts_stress" in obs_pre_filter.columns:
        stress_removed = obs_pre_filter.loc[_removed_idx, "pct_counts_stress"].median()
        stress_kept = obs_pre_filter.loc[_kept_idx, "pct_counts_stress"].median()
        enrichment["stress(median_pct)"] = {
            "在被过滤细胞中": f"{stress_removed:.2f}%",
            "在保留细胞中": f"{stress_kept:.2f}%",
            "富集倍数": round(stress_removed / max(stress_kept, 0.001), 1)
        }
    if "log_complexity" in obs_pre_filter.columns:
        cx_removed = obs_pre_filter.loc[_removed_idx, "log_complexity"].median()
        cx_kept = obs_pre_filter.loc[_kept_idx, "log_complexity"].median()
        enrichment["complexity(median)"] = {
            "在被过滤细胞中": f"{cx_removed:.3f}",
            "在保留细胞中": f"{cx_kept:.3f}",
            "富集倍数": "N/A"
        }
    
    if enrichment:
        display(pd.DataFrame(enrichment).T)
        print("\n解读：富集倍数 > 3 = MAD 过滤已隐含覆盖该标记物；≈ 1 = 两者独立，下游需额外处理")
    
    del obs_pre_filter  # 释放内存
else:
    print("无细胞被过滤，跳过交叉诊断")


## Per-sample 过滤影响


In [ ]:
# === Per-sample 过滤影响表 ===
if PER_SAMPLE_MAD and "filter_per_sample_stats" not in dir():
    pass  # 已通过 per-sample 阈值实现，此处汇总过滤后细胞分布

print("===== Per-sample 过滤影响 =====")
if "qc_report_v1" in adata.uns:
    print(f"总计: {adata.uns['qc_report_v1']['cells_before']:,} -> {adata.uns['qc_report_v1']['cells_after']:,} "
          f"(去除 {adata.uns['qc_report_v1']['pct_removed']}%)")
# 每个 sample 的细胞数（过滤后）
sample_counts = adata.obs["sample_id"].value_counts().sort_index()
print(sample_counts.to_string())
total = sample_counts.sum()
print(f"\n各 sample 占比:")
for sid, n in sample_counts.items():
    print(f"  {sid}: {n:,} ({100*n/total:.1f}%)")
min_sample = sample_counts.idxmin()
max_sample = sample_counts.idxmax()
if sample_counts.max() / max(sample_counts.min(), 1) > 5:
    print(f"\n⚠️ 样本间细胞数差异 > 5 倍（{max_sample}={sample_counts.max()} vs {min_sample}={sample_counts.min()}）")
    print("  -> 下游整合时可能需要考虑下采样平衡（02_merged 的 DOWNSAMPLE_TO_MIN）")


## 过滤前后对比

复刻过滤前的 QC 图，供 PI 做直观对比。

**对比检查要点**：
- 小提琴图：各指标的分布尾部是否被正确截断
- 散点图：被移除的细胞是否集中在预期区域（低基因数 + 高 MT% 区）
- 剩余细胞数是否合理：通常保留 80-95%

In [ ]:
# 过滤后 QC 小提琴图
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, metric in enumerate(["n_genes", "total_counts", "pct_counts_mt"]):
    ax = axes[i]
    sc.pl.violin(adata, keys=metric, groupby="sample_id", rotation=45, ax=ax, show=False)
    ax.set_title(f"{metric}（过滤后）")
plt.tight_layout()
fig.savefig("results/figures/01_kim_qc_violin_post.png", dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc.pl.scatter(adata, x="total_counts", y="n_genes", color="pct_counts_mt", ax=axes[0], show=False)
axes[0].set_title("total_counts vs n_genes（按 pct_mt 着色，过滤后）")
sc.pl.scatter(adata, x="n_genes", y="pct_counts_mt", color="total_counts", ax=axes[1], show=False)
axes[1].set_title("n_genes vs pct_mt（按 total_counts 着色，过滤后）")
plt.tight_layout()
fig.savefig("results/figures/01_kim_qc_scatter_post.png", dpi=150, bbox_inches="tight")
plt.show()


## QC 报告摘要

汇总本次 QC 的全部参数与结果，写入 `adata.uns["qc_report_v1"]`，供下游 notebook 读取。

In [ ]:
# QC 报告摘要
n_removed = cells_before - cells_after
qc_report = {
    "strategy": QC_STRATEGY,
    "n_mad": N_MAD if QC_STRATEGY == "adaptive" else None,
    "thresholds": {k: {kk: vv for kk, vv in v.items() if vv is not None} for k, v in thresholds.items()},
    "cells_before": int(cells_before),
    "cells_after": int(cells_after),
    "cells_removed": int(n_removed),
    "pct_removed": round(100 * n_removed / cells_before, 1) if cells_before > 0 else 0,
    "pre_qc_stats": qc_pre_stats,
    "doublet_rate_pct": round(100 * n_doublets_total / cells_before, 2) if cells_before > 0 else 0,
    "soupx_applied": soupx_applied,
    "soupx_cells_corrected": n_soupx_corrected,
    "cell_cycle_scored": SCORE_CELL_CYCLE,
    "flag_hb": FLAG_HEMOGLOBIN,
    "n_hb_flagged": globals().get("n_hb_flagged", 0),
}
adata.uns["qc_report_v1"] = qc_report
print("===== QC 报告摘要 =====")
for k, v in qc_report.items():
    print(f"  {k}: {v}")


In [ ]:
# Checkpoint：写入 per-dataset h5ad
# 先计算所有可见门禁；只有到保存阶段才占用 RUN_ID。

# F3修复：基因 ID 轴统一性断言——检查 per-dataset 内基因名无大小写混用
# 过大写/小写混用在 merge 时会导致 inner join 基因交集意外坍塌
_gene_names = list(adata.var_names)
_upper_count = sum(1 for g in _gene_names if g[0].isupper()) if _gene_names else 0
_lower_count = sum(1 for g in _gene_names if g[0].islower()) if _gene_names else 0
_total = len(_gene_names)
if _upper_count > 0 and _lower_count > 0:
    raise ValueError(
        f"基因 ID 轴不一致：{_upper_count} 个大写首字母基因 + {_lower_count} 个小写首字母基因 共 {_total} 个。"
        f"请统一基因名大小写（如全部 .str.upper()）后再进入 merge。"
    )
print(f"基因 ID 轴一致性检查通过：{_total} 个基因，统一为{'大写' if _upper_count > 0 else '小写'}首字母")

# ---- expression_contract 校验（决策 1/2）----
# 1. counts 获取与 shape 对齐校验
_counts = adata.layers[adata.uns["expression_contract"]["counts_layer"]]
assert _counts.shape == adata.shape, (
    f"counts layer shape {_counts.shape} != adata shape {adata.shape}"
)
# 2. 非负校验
if sp.issparse(_counts):
    _counts_data = _counts.data
else:
    _counts_data = _counts
assert (_counts_data >= 0).all(), "counts layer contains negative values"
# 3. 近整数校验（允许浮点误差 < 1e-6）
_diff = np.abs(_counts_data - np.round(_counts_data))
assert (_diff < 1e-6).all(), (
    f"counts layer contains non-integer values (max diff={_diff.max():.2e})"
)
# 4. 校验通过：更新契约
adata.uns["expression_contract"]["counts_validated"] = True
adata.uns["expression_contract"]["counts_integer_check"] = "full"
# 5. schema 校验
_contract = validate_expression_contract(adata)
print("expression_contract 校验通过: counts_validated=True, counts_integer_check=full")
print(f"基因 ID 轴一致性检查通过：{_total} 个基因，统一为{'大写' if _upper_count > 0 else '小写'}首字母")

_source_values = sorted(map(str, adata.obs["source_dataset"].dropna().unique())) if "source_dataset" in adata.obs.columns else []
hard_postconditions = {
    "non_empty": adata.n_obs > 0 and adata.n_vars > 0,
    "x_sparse_float32": sp.issparse(adata.X) and adata.X.dtype == np.float32,
    "source_dataset_unique": len(_source_values) == 1 and not adata.obs["source_dataset"].isna().any(),
    "source_matches_manifest": _source_values == [str(source_dataset)],
    "counts_layer_exists": "counts" in adata.layers,
    "counts_shape_aligned": _counts.shape == adata.shape,
    "counts_non_negative": True,
    "counts_integer_check_completed": adata.uns["expression_contract"]["counts_integer_check"] == "full",
    "counts_contract_validated": adata.uns["expression_contract"]["counts_validated"],
    # ---- doublet 三态 postconditions（计算完成的硬事实，非 needs_review）----
    # 仅在 doublet 检测已运行（doublet_class 列存在）时才验证 doublet postconditions
    # 未运行 doublet 的场景（如 shared test 合成 adata）doublet 相关键设为 True 跳过
}
_hd = "doublet_class" in adata.obs.columns
hard_postconditions["doublet_columns_present"] = all(
    c in adata.obs for c in ["doublet_score", "doublet_class", "doublet_include", "predicted_doublet"]
) if _hd else True
hard_postconditions["doublet_contract_present"] = "doublet_contract" in adata.uns if _hd else True
hard_postconditions["doublet_class_valid"] = (
    set(adata.obs["doublet_class"].unique()) <= {"singlet", "uncertain", "doublet"}
) if _hd else True
hard_postconditions["doublet_include_consistent"] = (
    not adata.obs.loc[adata.obs["doublet_class"] == "doublet", "doublet_include"].any()
) if _hd else True
_doublet_needs_review = bool(globals().get("doublet_needs_review", False))
stage_status = determine_stage_status({}, hard_postconditions, needs_review=_doublet_needs_review, allow_no_required_methods=True)
effective_parameters = snapshot_effective_parameters(globals(), exclude=("RSCRIPT_BIN", "R_AVAILABLE"), path_root=Path(_root))
runtime_provenance = collect_runtime_provenance(_root, ("anndata", "scanpy", "numpy", "pandas", "scipy"))
manifest_sha256 = sha256_file(MANIFEST_PATH)
run_paths = prepare_run(RUN_ROOT, RUN_ID)
manifest_payload = {
    "run_id": RUN_ID, "stage": "01_qcd", "stage_status": stage_status.value,
    "source_dataset": str(source_dataset),
    "inputs": [{"path": MANIFEST_PATH, "sha256": manifest_sha256}],
    "effective_parameters": effective_parameters, "runtime_provenance": runtime_provenance,
    "hard_postconditions": hard_postconditions,
}
if stage_status.value == "FAILED":
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise RuntimeError(f"Stage 01 FAILED: {hard_postconditions}")
adata.uns["stage"] = "01_qcd"
adata.uns["status"] = stage_status.value
adata.uns["upstream"] = [MANIFEST_PATH]
adata.uns["version"] = f"v{OUTPUT_VERSION}"
adata.uns["run_id"] = RUN_ID
draft_checkpoint = run_paths.draft_dir / OUTPUT_FILENAME
try:
    adata.write_h5ad(draft_checkpoint, compression="lzf")
    checkpoint_sha256 = sha256_file(draft_checkpoint)
except Exception as error:
    draft_checkpoint.unlink(missing_ok=True)
    manifest_payload["stage_status"] = "FAILED"
    manifest_payload["failure"] = {"type": type(error).__name__, "message": str(error)}
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise
manifest_payload["checkpoint"] = {"path": OUTPUT_FILENAME, "sha256": checkpoint_sha256}
atomic_write_json(run_paths.manifest_path, manifest_payload)
if stage_status.value == "NEEDS_REVIEW":
    # doublet 异常，只写 draft checkpoint，不提升为正式 checkpoint
    if _hd:
        manifest_payload["doublet_summary"] = {
            "needs_review": _doublet_needs_review,
            "n_excluded": int((~adata.obs["doublet_include"]).sum()),
            "n_doublet": int((adata.obs["doublet_class"] == "doublet").sum()),
            "n_uncertain": int((adata.obs["doublet_class"] == "uncertain").sum()),
            "n_singlet": int((adata.obs["doublet_class"] == "singlet").sum()),
            "reasons": globals().get("needs_review_reasons", []),
        }
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    OUTPUT_PATH = str(run_paths.draft_dir / OUTPUT_FILENAME)
    print(f"[NEEDS_REVIEW] doublet 检测异常，draft checkpoint 已写入 {OUTPUT_PATH}，未提升正式 checkpoint")
    print(f"  请 PI 审阅 per-sample 诊断表后再决定是否手动 promote 或调整阈值重跑。")
    if "doublet_summary" in manifest_payload:
        print(f"  排除细胞数={manifest_payload['doublet_summary']['n_excluded']}"
          f"  doublet={manifest_payload['doublet_summary']['n_doublet']}"
              f"  uncertain={manifest_payload['doublet_summary']['n_uncertain']}")
elif stage_status.value == "FAILED":
    # 保持现有 FAILED raise 逻辑不变（注意：此行在 FAILED 检测前已执行，此处只做结构占位）
    raise RuntimeError(f"Stage 01 FAILED: {hard_postconditions}")
else:
    OUTPUT_PATH = str(promote_run(run_paths))
    print(f"OK 提升 {OUTPUT_PATH}  ({adata.n_obs} cells x {adata.n_vars} genes)")


# per_dataset schema 校验
from scrna_integration.per_dataset_schema import validate_per_dataset_output
_schema_result = validate_per_dataset_output(adata)
if not _schema_result["passed"]:
    print("per_dataset schema 校验 FAILED:")
    for e in _schema_result["errors"]:
        print(f"  [ERROR] {e}")
else:
    print("per_dataset schema 校验 PASSED")
if _schema_result["warnings"]:
    for w in _schema_result["warnings"]:
        print(f"  [WARN] {w}")
gc.collect()
del adata; gc.collect()
print("内存已释放。")
